# 07 - Blinded response evaluation: reference validation

This notebook documents the reference validation stage of `docs/generation-response-evaluation-protocol.md`. We work with the approved reference set for 20 benchmark questions and three documented source discrepancies. We fingerprint and freeze the reference package before looking at generated responses.

The 20 reference answers, their acceptance rules and the three source-discrepancy decisions are approved.

We keep the reference answers and review form in the Git-ignored `data/raw/ground_truth/` folder. The notebook records counts, states and SHA-256 fingerprints without displaying private solution text. We keep exact private copies of the three original files before updating them, and later record the fingerprints in a public audit note.

The three write flags start as `False`. Each transition has a read-only preview before we enable its write flag. We keep generated responses outside this stage.

In [34]:
from __future__ import annotations

# Import standard-library tools for exact byte fingerprints and private-file updates.
import copy
import hashlib
import json
import os
import tempfile
from datetime import datetime, timezone
from pathlib import Path


# Keep all three transitions disabled until their preceding checks are reviewed.
RUN_RECORD_APPROVAL = False
RUN_FREEZE_REFERENCE = False
RUN_WRITE_PUBLIC_AUDIT = False

# Resolve the project root from either the repository root or its notebooks folder.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "configs/generation-response-evaluation-config.json").is_file():
    raise FileNotFoundError(f"Thesis repository not found at {PROJECT_ROOT}")

# Keep the proposed form private, beside the existing private reference records.
PRIVATE_DIR = PROJECT_ROOT / "data/raw/ground_truth"
REVIEW_FORM = PRIVATE_DIR / "Geotechnical_reference_answer_expert_review.docx"
ANSWERS = PRIVATE_DIR / "benchmark-reference-answers.jsonl"
DISCREPANCIES = PRIVATE_DIR / "benchmark-reference-source-discrepancies.jsonl"
VALIDATION = PRIVATE_DIR / "benchmark-reference-validation.json"

# These fingerprints were checked before the professor's approval was recorded.
ORIGINAL_SHA256 = {
    ANSWERS.name: "e51162fdde2884a8e6f59219c9d4f68db4e01a791d524db7520695984b37e6d2",
    DISCREPANCIES.name: "55babe7c64ad94a84e2fcff701a23d0bc2eab113d76ec9cfe88a14edf53a1613",
    VALIDATION.name: "e5e63c9b8fc34f112c6f32ef994447efc6e0e3356ff921c41544a853466e4ebd",
}
APPROVED_SHA256 = {
    ANSWERS.name: "981650c4f1cdf622bffaec0f84f263105b5ff1b2f6a41ecb273577c9410cb5c8",
    DISCREPANCIES.name: "7ba6ccc135f5354c85fff60b60781b5700691abe8f76477779f72ffef73e8571",
    VALIDATION.name: "962f274bd4009a47ea0349691000e7c778fc1a8cf36b6e2c67434a1d1d4e46d1",
}
REVIEW_SHA256 = "0faaccf8a0d79dfb0426c3a3b3b2ed73ca68b4e57e9f0d25096deea4488da1cf"
APPROVAL_REPORTED_DATE = "2026-09-24"
SOURCE_LINKAGE_COMMIT = "0691263c28b31f5376be18e50c2bba316f0c1d04"


def sha256_file(path: Path) -> str:
    """Fingerprint exact bytes, including whitespace and newline characters."""
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def sha256_bytes(content: bytes) -> str:
    """Fingerprint planned bytes before any private file is replaced."""
    return hashlib.sha256(content).hexdigest()


def require(condition: bool, explanation: str) -> None:
    """Stop a transition when a frozen input or review gate differs."""
    if not condition:
        raise ValueError(f"STOP: {explanation}")


print("Project root:", PROJECT_ROOT)
print("Write flags:", RUN_RECORD_APPROVAL, RUN_FREEZE_REFERENCE, RUN_WRITE_PUBLIC_AUDIT)

Project root: /home/zaki/coding_F/GismaProjects/Thesis
Write flags: False False False


## 1. Check the inputs and blinding state

We first identify whether the reference package is still at the original reconstruction, the approved version, or the frozen version. We compare exact file fingerprints, verify the frozen question set, configuration and solutions manual, and check the 20 reference records and three discrepancy records. Generated responses and private solution text are not displayed.

The original validation metadata still contains an old discrepancy count and a stale hash alias. We record that difference here and correct both fields in the approval update, so the change remains visible in the audit trail.

In [35]:
def read_jsonl(path: Path) -> list[dict]:
    """Read each complete private record and reject blank or malformed lines."""
    lines = path.read_text(encoding="utf-8").splitlines()
    require(bool(lines) and all(line.strip() for line in lines), f"blank JSONL line: {path.name}")
    return [json.loads(line) for line in lines]


def read_reference_state() -> tuple[str, dict, list[dict], list[dict]]:
    """Recognise the original, approved or frozen state without opening responses."""
    # Require the exact proposal reviewed by the professor.
    require(REVIEW_FORM.is_file(), f"missing review form: {REVIEW_FORM}")
    require(sha256_file(REVIEW_FORM) == REVIEW_SHA256, "review form differs")

    # Read and count only the three private reference files.
    for path in (ANSWERS, DISCREPANCIES, VALIDATION):
        require(path.is_file(), f"missing reference file: {path}")
    metadata = json.loads(VALIDATION.read_text(encoding="utf-8"))
    answers = read_jsonl(ANSWERS)
    discrepancies = read_jsonl(DISCREPANCIES)
    require(len(answers) == 20 and len(discrepancies) == 3, "expected 20 answers and 3 discrepancies")
    require([row["position"] for row in answers] == list(range(20)), "answer positions changed")
    require(len({row["discrepancy_id"] for row in discrepancies}) == 3, "duplicate discrepancy ID")
    require(metadata["source_linkage_commit"] == SOURCE_LINKAGE_COMMIT, "source linkage changed")
    require(metadata["benchmark_dataset_id"] == "tophel-2025-challenging-20-reconstruction-v1", "dataset changed")

    # Keep the agreed blind barrier closed throughout the reference phase.
    for key in ("generated_response_file_opened", "response_text_loaded", "human_response_scoring_started"):
        require(metadata[key] is False, f"blinding flag changed: {key}")

    # Verify source files against metadata whose provenance is already fingerprinted.
    for path_key, hash_key in (
        ("question_file_relative_path", "question_file_sha256"),
        ("question_config_relative_path", "question_config_sha256"),
        ("solution_source_relative_path", "solution_source_document_sha256"),
    ):
        source_path = PROJECT_ROOT / metadata[path_key]
        require(source_path.is_file(), f"missing source: {metadata[path_key]}")
        require(sha256_file(source_path) == metadata[hash_key], f"source changed: {metadata[path_key]}")
    question_rows = read_jsonl(PROJECT_ROOT / metadata["question_file_relative_path"])
    require([row["question_id"] for row in answers] == [row["question_id"] for row in question_rows],
            "answer IDs or order differ from the frozen questions")

    # Classify by independent file hashes before accepting metadata's own status.
    current = {path.name: sha256_file(path) for path in (ANSWERS, DISCREPANCIES, VALIDATION)}
    if current == ORIGINAL_SHA256:
        state = "pre_approval"
        require(metadata["pending_expert_validation_count"] == 20, "original approval count changed")
    elif current == APPROVED_SHA256:
        state = "approved"
        require(metadata["expert_validation_complete"] is True, "approval flag missing")
        require(metadata["references_frozen_for_scoring"] is False, "unexpected freeze flag")
    elif (current[ANSWERS.name] == APPROVED_SHA256[ANSWERS.name]
          and current[DISCREPANCIES.name] == APPROVED_SHA256[DISCREPANCIES.name]
          and metadata.get("references_frozen_for_scoring") is True
          and metadata.get("reference_answers_ready_for_response_scoring") is True):
        state = "frozen"
        require(metadata["reference_answer_sha256"] == current[ANSWERS.name], "answer pointer mismatch")
        require(metadata["source_discrepancy_log_sha256"] == current[DISCREPANCIES.name], "discrepancy pointer mismatch")
    else:
        raise ValueError("STOP: reference files do not match a recognised validation state")
    return state, metadata, answers, discrepancies


# Report only public-safe state, counts and fingerprints.
state, metadata, answer_rows, discrepancy_rows = read_reference_state()
print("Reference state:", state)
for path in (ANSWERS, DISCREPANCIES, VALIDATION, REVIEW_FORM):
    print(path.name, sha256_file(path))
print("Reference records:", len(answer_rows))
print("Discrepancy records:", len(discrepancy_rows))
print("Frozen question set, config and manual: VERIFIED")
print("Generated responses accessed: False")
if state == "pre_approval":
    print("Known stale discrepancy-count alias:", metadata["source_discrepancy_count"])
    print("Known stale discrepancy-hash alias:", metadata["source_discrepancy_log_sha256"] != sha256_file(DISCREPANCIES))

Reference state: frozen
benchmark-reference-answers.jsonl 981650c4f1cdf622bffaec0f84f263105b5ff1b2f6a41ecb273577c9410cb5c8
benchmark-reference-source-discrepancies.jsonl 7ba6ccc135f5354c85fff60b60781b5700691abe8f76477779f72ffef73e8571
benchmark-reference-validation.json dc6fd4b4ba725a2517cd8e8bb5ed6b095584571e050dd349b5583fa225bfd356
Geotechnical_reference_answer_expert_review.docx 0faaccf8a0d79dfb0426c3a3b3b2ed73ca68b4e57e9f0d25096deea4488da1cf
Reference records: 20
Discrepancy records: 3
Frozen question set, config and manual: VERIFIED
Generated responses accessed: False


## 2. Prepare the approved reference package

The approved reference set includes the expected results, accepted methods and alternatives, numerical tolerances, and three source-discrepancy decisions. We construct the updated files in memory first. Only the review status and audit metadata change; the calculations and acceptance rules stay the same.

We compare the planned files with their expected SHA-256 fingerprints before replacing any private file. This checks that the approval update has not changed the mathematical content or file format by accident.

In [36]:
def encode_jsonl(records: list[dict]) -> bytes:
    """Serialize JSONL deterministically without changing parsed answer values."""
    return ("\n".join(json.dumps(row, ensure_ascii=False, sort_keys=True) for row in records) + "\n").encode("utf-8")


def encode_json(record: dict) -> bytes:
    """Serialize readable metadata with exactly one final newline."""
    return (json.dumps(record, ensure_ascii=False, sort_keys=True, indent=2) + "\n").encode("utf-8")


def build_approved_files() -> dict[Path, bytes]:
    """Build the three exact approved files in memory from frozen originals."""
    require(state == "pre_approval", "approval build requires the original state")
    for path in (ANSWERS, DISCREPANCIES, VALIDATION):
        require(sha256_file(path) == ORIGINAL_SHA256[path.name], f"original changed: {path.name}")
    require(all(row["validation_status"] == "pending" for row in answer_rows), "answer already reviewed")
    require(all(row["gradable"] is True for row in answer_rows), "question eligibility changed")

    # Work on deep copies so the read-only notebook state is not silently mutated.
    answers = copy.deepcopy(answer_rows)
    discrepancies = copy.deepcopy(discrepancy_rows)
    approved_metadata = copy.deepcopy(metadata)
    original_notes = {row["question_id"]: row["validation_note"] for row in answers}

    # Distinguish the researcher's report from a signed or returned review form.
    approval_record = {
        "approval_scope": "All 20 solutions, their acceptance rules, and all three discrepancy decisions accepted without changes.",
        "approved_review_document_sha256": REVIEW_SHA256,
        "evidence_type": "researcher_reported_professor_approval",
        "original_answer_validation_notes": original_notes,
        "pre_approval_file_sha256": ORIGINAL_SHA256,
        "professor_name": "Professor Peña Olarte",
        "professor_review_date": None,
        "reported_on_utc_date": APPROVAL_REPORTED_DATE,
        "returned_annotated_or_signed_copy_received": False,
        "reviewer_role": "geotechnical_domain_expert",
    }

    # Mark all 20 records while leaving values, methods and tolerances intact.
    for row in answers:
        row["validation_status"] = "validated"
        row["validated_by_role"] = "geotechnical_domain_expert"
        row["validation_note"] = (
            "Confirmed without changes by Professor Peña Olarte; approval "
            f"reported by the thesis researcher on {APPROVAL_REPORTED_DATE}. See expert_approval_record."
        )

    # Confirm the three distinct discrepancy records without rewriting their resolution.
    for row in discrepancies:
        status_key = ("domain_expert_confirmation_status" if "domain_expert_confirmation_status" in row
                      else "professor_confirmation_status")
        require(row[status_key] == "pending", "discrepancy already reviewed")
        row[status_key] = "confirmed"
        row["confirmed_by_role"] = "geotechnical_domain_expert"
        row["confirmation_note"] = (
            "Approved without changes by Professor Peña Olarte, as reported "
            f"by the thesis researcher on {APPROVAL_REPORTED_DATE}."
        )

    # Fingerprint the two updated JSONL files before repairing stale metadata fields.
    answer_bytes = encode_jsonl(answers)
    discrepancy_bytes = encode_jsonl(discrepancies)
    approved_metadata.update({
        "all_gradable_records_validated": True,
        "discrepancy_log_sha256": sha256_bytes(discrepancy_bytes),
        "expert_approval_record": approval_record,
        "expert_validation_complete": True,
        "last_expert_validation_update_on_utc": APPROVAL_REPORTED_DATE,
        "pending_expert_validation_count": 0,
        "pending_record_count": 0,
        "reference_answer_sha256": sha256_bytes(answer_bytes),
        "reference_answers_ready_for_expert_validation": False,
        "reference_answers_ready_for_response_scoring": False,
        "references_frozen_for_scoring": False,
        "source_discrepancy_confirmed_count": 3,
        "source_discrepancy_count": 3,
        "source_discrepancy_log_sha256": sha256_bytes(discrepancy_bytes),
        "source_discrepancy_pending_confirmation_count": 0,
        "validated_record_count": 20,
    })
    for group in approved_metadata.get("reconstruction_groups", []):
        if group.get("status") == "reconstructed_pending_domain_expert_validation":
            group["status"] = "reconstructed_domain_expert_validated"

    # Compare every planned byte string to the previously audited output.
    planned = {
        ANSWERS: answer_bytes,
        DISCREPANCIES: discrepancy_bytes,
        VALIDATION: encode_json(approved_metadata),
    }
    for path, content in planned.items():
        require(sha256_bytes(content) == APPROVED_SHA256[path.name], f"unexpected approval output: {path.name}")
    return planned


# Preview output fingerprints only; do not write anything in this cell.
if state == "pre_approval":
    approved_preview = build_approved_files()
    for path, content in approved_preview.items():
        print("Proposed approved fingerprint:", path.name, sha256_bytes(content))
    print("Approval preview: PASSED; files unchanged")
else:
    print("Approval preview skipped; current state:", state)

Approval preview skipped; current state: frozen


## 3. Record the approval

The approval transition uses `RUN_RECORD_APPROVAL`. It checks the original fingerprints again, saves exact private copies of the three original records, and then replaces those records with the approved versions. The transition only works from the original state, which prevents an accidental second approval update.

These private records remain Git-ignored. The backup files preserve the original bytes, while the notebook records the validation logic and fingerprints.

In [37]:
def backup_original(path: Path, expected_hash: str, suffix: str) -> Path:
    """Retain one exact private backup and refuse to overwrite a different one."""
    backup = path.with_name(f"{path.stem}.{suffix}{path.suffix}")
    content = path.read_bytes()
    require(sha256_bytes(content) == expected_hash, f"source changed before backup: {path.name}")
    if backup.exists():
        require(sha256_file(backup) == expected_hash, f"existing backup differs: {backup.name}")
    else:
        with backup.open("xb") as target:
            target.write(content)
    return backup


def replace_exact_bytes(path: Path, content: bytes) -> None:
    """Stage a complete file beside its target before replacing that target."""
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(mode="wb", dir=path.parent, prefix=".reference-stage-", delete=False) as staged:
            temporary_path = Path(staged.name)
            staged.write(content)
            staged.flush()
            os.fsync(staged.fileno())
        os.replace(temporary_path, path)
    finally:
        if temporary_path is not None and temporary_path.exists():
            temporary_path.unlink()


if RUN_RECORD_APPROVAL:
    require(state == "pre_approval", "approval may only run once from the original state")
    planned = build_approved_files()
    for path in (ANSWERS, DISCREPANCIES, VALIDATION):
        backup = backup_original(path, ORIGINAL_SHA256[path.name], "pre-expert-approval")
        print("Private original backup:", backup.name, sha256_file(backup))
    for path, content in planned.items():
        replace_exact_bytes(path, content)
        require(sha256_file(path) == APPROVED_SHA256[path.name], f"post-write hash differs: {path.name}")
        print("Approved:", path.name, sha256_file(path))
    state, metadata, answer_rows, discrepancy_rows = read_reference_state()
    require(state == "approved", "approval did not reach the approved state")
    print("Expert approval recorded; response scoring remains closed.")
else:
    print("Approval write: DISABLED")

Approval write: DISABLED


## 4. Check the approved package

We verify that all 20 answers have a validated status and all three discrepancy decisions are confirmed. We also check the internal counts and hash fields. By comparing each approved record with its original private copy, we confirm that source values, methods, alternatives, units and tolerances have not changed. Response scoring is still closed at this point.

In [38]:
if state in ("approved", "frozen"):
    # Confirm status and role for every approved answer and discrepancy record.
    require(all(row["validation_status"] == "validated" for row in answer_rows), "answer not validated")
    require(all(row["validated_by_role"] == "geotechnical_domain_expert" for row in answer_rows), "reviewer role changed")
    for row in discrepancy_rows:
        status = row.get("domain_expert_confirmation_status", row.get("professor_confirmation_status"))
        require(status == "confirmed", "discrepancy not confirmed")

    # Verify that both established hash aliases and all counts now agree.
    require(metadata["reference_answer_sha256"] == sha256_file(ANSWERS), "answer hash pointer differs")
    require(metadata["source_discrepancy_log_sha256"] == sha256_file(DISCREPANCIES), "discrepancy hash pointer differs")
    require(metadata["discrepancy_log_sha256"] == sha256_file(DISCREPANCIES), "discrepancy hash alias differs")
    require(metadata["source_discrepancy_count"] == 3, "stale discrepancy count remains")
    require(metadata["validated_record_count"] == 20, "validated count differs")
    require(metadata["source_discrepancy_confirmed_count"] == 3, "confirmed count differs")

    # Compare the private answers and discrepancies with their exact original backups.
    original_answers = read_jsonl(ANSWERS.with_name(f"{ANSWERS.stem}.pre-expert-approval{ANSWERS.suffix}"))
    original_discrepancies = read_jsonl(DISCREPANCIES.with_name(
        f"{DISCREPANCIES.stem}.pre-expert-approval{DISCREPANCIES.suffix}"
    ))
    for original, approved in zip(original_answers, answer_rows, strict=True):
        for key in ("validation_status", "validated_by_role", "validation_note"):
            original.pop(key, None)
            approved = {name: value for name, value in approved.items() if name != key}
        require(original == approved, "answer content changed beyond review fields")
    for original, approved in zip(original_discrepancies, discrepancy_rows, strict=True):
        for key in ("domain_expert_confirmation_status", "professor_confirmation_status",
                    "confirmed_by_role", "confirmation_note"):
            original.pop(key, None)
            approved = {name: value for name, value in approved.items() if name != key}
        require(original == approved, "discrepancy resolution changed beyond review fields")

    print("Approved answer records: 20; discrepancy decisions: 3")
    print("Original answer content and discrepancy resolutions: UNCHANGED")
    print("Reference frozen for scoring:", metadata["references_frozen_for_scoring"])
else:
    print("Approval audit pending; current state:", state)

Approved answer records: 20; discrepancy decisions: 3
Original answer content and discrepancy resolutions: UNCHANGED
Reference frozen for scoring: True


## 5. Freeze the reference package

We freeze the reference only after the approved answers, discrepancy decisions, review-form fingerprint and frozen source files pass their checks. This transition changes the private validation metadata; the approved answers and discrepancy records keep the same bytes. We record the freeze date in UTC.

`RUN_FREEZE_REFERENCE` controls this step. It remains off while we inspect the planned metadata fingerprint. The generated responses stay outside the notebook until the freeze and public audit are complete.

In [39]:
def build_frozen_metadata() -> bytes:
    """Construct the final private metadata after every expert and source gate passes."""
    require(state == "approved", "freeze requires the approved, unfrozen state")
    require(sha256_file(ANSWERS) == APPROVED_SHA256[ANSWERS.name], "approved answers changed")
    require(sha256_file(DISCREPANCIES) == APPROVED_SHA256[DISCREPANCIES.name], "approved discrepancies changed")
    require(sha256_file(VALIDATION) == APPROVED_SHA256[VALIDATION.name], "approved metadata changed")
    require(metadata["expert_approval_record"]["approved_review_document_sha256"] == REVIEW_SHA256,
            "approved review form pointer differs")
    require(metadata["expert_validation_complete"] is True, "expert approval incomplete")
    require(metadata["source_discrepancy_count"] == metadata["source_discrepancy_record_count"] == 3,
            "discrepancy counts conflict")
    require(metadata["validated_record_count"] == 20, "20 validated answers required")
    require(all(row["validation_status"] == "validated" for row in answer_rows), "pending answer remains")
    require(all(row.get("domain_expert_confirmation_status", row.get("professor_confirmation_status")) == "confirmed"
                for row in discrepancy_rows), "pending discrepancy remains")

    # The read-only preflight also verified all three frozen source fingerprints.
    frozen_metadata = copy.deepcopy(metadata)
    frozen_metadata["reference_answers_ready_for_response_scoring"] = True
    frozen_metadata["references_frozen_for_scoring"] = True
    frozen_metadata["references_frozen_on_utc_date"] = datetime.now(timezone.utc).date().isoformat()
    return encode_json(frozen_metadata)


if state == "approved":
    freeze_preview = build_frozen_metadata()
    print("Proposed frozen metadata fingerprint:", sha256_bytes(freeze_preview))
    print("Answers and discrepancy files remain unchanged")
else:
    print("Freeze preview skipped; current state:", state)

Freeze preview skipped; current state: frozen


In [40]:
if RUN_FREEZE_REFERENCE:
    require(state == "approved", "freeze may only run once after approval")
    frozen_bytes = build_frozen_metadata()
    backup = backup_original(VALIDATION, APPROVED_SHA256[VALIDATION.name], "pre-reference-freeze")
    print("Private approved-metadata backup:", backup.name, sha256_file(backup))
    replace_exact_bytes(VALIDATION, frozen_bytes)
    require(sha256_file(VALIDATION) == sha256_bytes(frozen_bytes), "freeze write differs")
    state, metadata, answer_rows, discrepancy_rows = read_reference_state()
    require(state == "frozen", "reference freeze did not complete")
    print("Frozen metadata fingerprint:", sha256_file(VALIDATION))
    print("References frozen. Do not open responses until the public audit is saved and committed.")
else:
    print("Reference freeze write: DISABLED")

Reference freeze write: DISABLED


## 6. Record the public fingerprints

We document the original, approved and frozen fingerprints in `docs/` after the reference package is frozen. This public note contains no benchmark question text, reference solution, source-discrepancy explanation, generated answer or evaluator note. We commit the note and notebook before starting blinded response scoring.

In [41]:
def public_audit_text() -> str:
    """Prepare a text-free, trackable record of the private validation gate."""
    require(state == "frozen", "public audit requires a frozen reference")
    require(sha256_file(ANSWERS) == APPROVED_SHA256[ANSWERS.name], "frozen answer changed")
    require(sha256_file(DISCREPANCIES) == APPROVED_SHA256[DISCREPANCIES.name], "frozen discrepancy changed")
    return f"""# Reference-answer expert validation audit

This audit records the frozen reference validation step in our generation-response evaluation. It contains no private benchmark question, reference answer, reviewer note or generated response.

## Expert decision and scope

- We recorded approval of the 20 reference answers, their acceptance rules and all three source-discrepancy decisions on {APPROVAL_REPORTED_DATE}. The private metadata stores the approval source.
- The review form SHA-256 is `{REVIEW_SHA256}`.
- We did not open generated response texts during reference validation.

## Private-file fingerprints

| Private file | Before approval SHA-256 | Approved/frozen SHA-256 |
| --- | --- | --- |
| `benchmark-reference-answers.jsonl` | `{ORIGINAL_SHA256[ANSWERS.name]}` | `{sha256_file(ANSWERS)}` |
| `benchmark-reference-source-discrepancies.jsonl` | `{ORIGINAL_SHA256[DISCREPANCIES.name]}` | `{sha256_file(DISCREPANCIES)}` |
| `benchmark-reference-validation.json` | `{ORIGINAL_SHA256[VALIDATION.name]}` | Approval: `{APPROVED_SHA256[VALIDATION.name]}`; final freeze: `{sha256_file(VALIDATION)}` |

The frozen question set SHA-256 is `{metadata['question_file_sha256']}`, its configuration SHA-256 is `{metadata['question_config_sha256']}`, and the solutions manual SHA-256 is `{metadata['solution_source_document_sha256']}`. The source-linkage commit is `{SOURCE_LINKAGE_COMMIT}`.

We keep private backups of the original files and approved metadata beside the working references. We froze the reference gate on {metadata['references_frozen_on_utc_date']}. We commit this public audit and the notebook before loading generated responses.
"""


if state == "frozen":
    audit_path = PROJECT_ROOT / "docs/reference-answer-expert-validation-audit.md"
    audit_content = public_audit_text()
    print("Public audit target:", audit_path.relative_to(PROJECT_ROOT))
    print("Proposed public audit SHA-256:", sha256_bytes(audit_content.encode("utf-8")))
    if RUN_WRITE_PUBLIC_AUDIT:
        # Refuse to replace a different reviewed audit with new content.
        if audit_path.exists():
            require(audit_path.read_text(encoding="utf-8") == audit_content,
                    "existing public audit differs; review before replacing")
        else:
            audit_path.write_text(audit_content, encoding="utf-8")
        print("Public audit saved:", audit_path.relative_to(PROJECT_ROOT))
    else:
        print("Public audit write: DISABLED")
else:
    print("Public audit pending until the reference is frozen")

Public audit target: docs/reference-answer-expert-validation-audit.md
Proposed public audit SHA-256: a15d2172e40ee5ae49aae503d81ed0575f3853d072f984c21f02971703279bf9
Public audit write: DISABLED


## 7. Prepare for blinded scoring

At this checkpoint, the approved reference answers and the public audit have been frozen. We create the separate private grouped scoring notebook through the offline terminal tool. Notebook 07 remains the record of reference validation and does not contain generated responses or response judgments.

We retain the nine answers without visible response text in the frozen 140-response set under the existing evaluation policy. Unblinding and condition-level results belong to the later analysis stage.
